# 3. A Rede de Monitoramento Brasileira


<div style="text-align: justify">Foram contabilizadas 479 estações de monitoramento da qualidade do ar no Brasil no ano de 2024, representando um acréscimo de 84 estações em relação ao levantamento realizado no ano de 2023 (BRASIL, 2023). Deste montante, 385 são estações que utilizam método de referência ou equivalente, um acréscimo de 27 unidades (aumento de 7,5%) em relação ao ano anterior. Já o monitoramento indicativo teve uma ampliação de 57 unidades, atingindo um total de 94 estações (aumento de 154%).</div><br/>
<div style="text-align: justify">O acréscimo significativo de unidades de monitoramento está associado ao maior número de respostas das UFs ao questionário aplicado pelo MMA, uma vez que identificaram mais estações de referência ou equivalentes que já operavam, mas não foram contabilizadas, e à implantação de estações de monitoramento indicativas. Considerando as respostas do questionário aplicado em 2023, apenas dois estados, Acre e Mato Grosso, afirmaram possuir um total de 37 estações de monitoramento indicativas instaladas. Enquanto isso, em 2024, mais 6 UFs (Amazonas, Amapá, Goiás, Maranhão, Pará e Tocantins) informaram a operação de estações indicativas, atingindo a marca de 94 equipamentos em operação.</div>

```{note}
É importante destacar que as estações indicativas são integradas em plataformas internacionais com finalidade científica, exploratória e informativa. A gestão dos dados ocorre através da integração dos equipamentos na plataforma, sem qualquer tratamento da informação. Em alguns casos, a operação e supervisão é realizada pelos OEMAs. Estas estações são equipadas com instrumentos e sensores não considerados equivalentes às estações de referência. As estações indicativas são capazes de monitorar a concentração de alguns poluentes atmosféricos em tempo real, no entanto, podem apresentar um grau de incerteza relevante em relação ao dado gerado, principalmente quando não são calibradas e operadas adequadamente. 
```

<div style="text-align: justify">Convém ressaltar, ainda, que as estações indicativas não atendem aos critérios estabelecidos pelo Guia Técnico para o Monitoramento e Avaliação da Qualidade do Ar, mas podem fornecer informações relevantes sobre a qualidade do ar, principalmente em locais sem nenhuma estação de referência ou equivalente.</div><br/>
<div style="text-align: justify">As Figuras iterativa abaixo mostram a evolução do número de pontos de monitoramento de cada poluente atmosférico por estado, a expansão ou retração da rede estadual em cada ano e o número de pontos de monitoramento por poluente que foram registrados na base de dados desenvolvida neste relatório. </div>
<br>
<br>

In [2]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import os
from IPython.display import HTML
import numpy as np


# Caminho para a pasta de dados
rootPath = os.path.dirname(os.getcwd())
aqmData = pd.read_csv(rootPath+'/data/Monitoramento_QAr_BR.csv')

# Explode anos monitorados
df_exploded = aqmData.explode("ANOS_MONITORADOS")

# Consolidate ANOS_MONITORADOS per UF and ID_MMA
aqmData_consolidated = (
    df_exploded.groupby(['UF','ID_MMA'])['ANOS_MONITORADOS']
    .apply(lambda x: ', '.join(sorted(set(map(str, x)))))
    .reset_index()
)
aqmData_consolidated.loc[aqmData_consolidated['ANOS_MONITORADOS']=='nan','ANOS_MONITORADOS'] = np.nan
aqmData_consolidated = aqmData_consolidated.dropna(subset=['ANOS_MONITORADOS'])


# Convert string of years to list of ints
def process_years(x):
    if pd.isna(x):
        return []
    years = [int(y.strip()) for y in str(x).split(',') if y.strip().isdigit()]
    return sorted(set(years))

aqmData_consolidated['ANOS_MONITORADOS'] = aqmData_consolidated['ANOS_MONITORADOS'].apply(process_years)

# Explode again to have one row per year
df_exploded_years = aqmData_consolidated.explode('ANOS_MONITORADOS')
df_exploded_years['ANOS_MONITORADOS'] = df_exploded_years['ANOS_MONITORADOS'].astype(int)

# Count stations per UF per year
df_counts = (
    df_exploded_years.groupby(['ANOS_MONITORADOS', 'UF'])['ID_MMA']
    .nunique()
    .reset_index(name='Num_Stations')
)
df_counts = df_counts[df_counts['ANOS_MONITORADOS'] <= 2024]
anos = sorted(df_counts['ANOS_MONITORADOS'].unique())
ufs = sorted(df_counts['UF'].unique())

# ---------------------------
# Figure 1: Stacked bars
# ---------------------------
# Common layout settings
layout_style = dict(
    plot_bgcolor="whitesmoke",   # light gray for plot area
    paper_bgcolor="white",  # light gray for outer background
    hovermode='x unified',
    legend_title="UF",
    height=600
)

fig_bars = go.Figure()
for uf in ufs:
    df_uf = df_counts[df_counts['UF'] == uf].sort_values('ANOS_MONITORADOS')
    fig_bars.add_trace(go.Bar(
        x=df_uf['ANOS_MONITORADOS'],
        y=df_uf['Num_Stations'],
        name=uf
    ))

fig_bars.update_layout(
    **layout_style,
    barmode='stack',
    title="Número de pontos de monitoramento por UF",
    xaxis_title="Ano",
    yaxis_title="Número de pontos",
    #legend_title="UF",
    #hovermode='x unified',
    #height=600,
    xaxis_range=[df_exploded_years['ANOS_MONITORADOS'].min() - 0.5, 2024.5],
    xaxis=dict(
        type='linear',
        title="Ano",
        tickmode='array',
        tickvals=anos,       # força mostrar todos os anos
        ticktext=anos
    ),
)

#fig_bars.show()
HTML(fig_bars.to_html(include_plotlyjs="cdn"))

In [ ]:
# ---------------------------
# Figure 2: Difference lines
# ---------------------------
layout_style = dict(
    plot_bgcolor="white",   # light gray for plot area
    paper_bgcolor="white",  # light gray for outer background
    hovermode='x unified',
    legend_title="UF",
    height=600
)

fig_diff = go.Figure()
difUFmin = []
difUFmax = []
for uf in ufs:
    df_uf = df_counts[df_counts['UF'] == uf].sort_values('ANOS_MONITORADOS')
    df_uf['Difference'] = df_uf['Num_Stations'].diff().fillna(0)
    fig_diff.add_trace(go.Scatter(
        x=df_uf['ANOS_MONITORADOS'],
        y=df_uf['Difference'],
        mode='lines+markers',
        name=uf
    ))
    difUFmin.append(df_uf['Difference'].min())
    difUFmax.append(df_uf['Difference'].max())

max_abs_value = max(max(difUFmin, key=abs),max(difUFmax, key=abs))

fig_diff.update_layout(
    **layout_style,
    title="Diferença anual de pontos de monitoramento por UF",
    xaxis_title="Ano",
    yaxis_title="Diferença anual",
    #legend_title="UF",
    #hovermode='x unified',
    #height=600,
    xaxis_range=[df_exploded_years['ANOS_MONITORADOS'].min() - 0.5, 2024.5],
    xaxis=dict(
        type='linear',
        title="Ano",
        tickmode='array',
        tickvals=anos,       # força mostrar todos os anos
        ticktext=anos
    ),
    shapes=[
        dict(
            type="rect",
            xref="paper", x0=0, x1=1,         # full width
            yref="y", y0=-max_abs_value-1, y1=0,  
            fillcolor="mistyrose",            # light red background
            opacity=0.5,
            layer="below",
            line_width=0
        ),
        dict(
            type="rect",
            xref="paper", x0=0, x1=1,         # full width
            yref="y", y0=max_abs_value+1, y1=0,  
            fillcolor="lightblue",            # light red background
            opacity=0.5,
            layer="below",
            line_width=0
        ),
        # Add thick horizontal line at y=0
        dict(
            type="line",
            xref="paper", x0=0, x1=1,  # full width of the plot
            yref="y", y0=0, y1=0,
            line=dict(color="white", width=6),  # thicker line
            layer="below"               # 👈 puts it behind all data
        )
    ]
)

# Display figures

#fig_diff.show()
HTML(fig_diff.to_html(include_plotlyjs="cdn"))

**apresentar figura por poluentes**

In [ ]:
# Caminho para a pasta de dados
rootPath = os.path.dirname(os.getcwd())

# Lendo o csv
aqmData = pd.read_csv(rootPath+'/data/Monitoramento_QAr_BR.csv')

df = aqmData.copy()

# Quebrar ANOS_MONITORADOS em listas
df["ANOS_MONITORADOS"] = df["ANOS_MONITORADOS"].str.split(",")

# To remove all spaces
df['UF'] = df['UF'].str.replace(" ", "")


# Explodir para cada ano virar uma linha
df_exploded = df.explode("ANOS_MONITORADOS")

# Converter para inteiro
#df_exploded["ANOS_MONITORADOS"] = df_exploded["ANOS_MONITORADOS"].astype(int)

# Agrupar também por POLUENTE
df_grouped = (
    df_exploded.groupby(["POLUENTE", "UF", "ANOS_MONITORADOS"])
    .size()   # conta linhas
    .reset_index(name="NSTATION")
)

df_grouped['ANOS_MONITORADOS'] = df_grouped['ANOS_MONITORADOS'].astype(int)



# Lista de poluentes e UFs
poluentes = sorted(df_grouped["POLUENTE"].unique())
ufs = sorted(df_grouped["UF"].unique())


# Cores fixas por UF
color_map = {uf: f"hsl({i*40 % 360},70%,50%)" for i, uf in enumerate(ufs)}

layout_style = dict(
    plot_bgcolor="whitesmoke",   # light gray for plot area
    paper_bgcolor="white",  # light gray for outer background
    hovermode='x unified',
    legend_title="UF",
    height=600
)
fig = go.Figure()
trace_visibility = []

## Track UF que já teve legenda
#uf_legend_shown = {uf: False for uf in ufs}

# Criar todos os traços: um trace por UF e poluente
for pol in poluentes:
    df_pol = df_grouped[df_grouped["POLUENTE"] == pol]
    for uf in ufs:
        df_uf = df_pol[df_pol["UF"] == uf]
        y_values = df_uf["NSTATION"].values
        x_values = df_uf["ANOS_MONITORADOS"].values 

        fig.add_trace(
            go.Bar(
                x=x_values,
                y=y_values,
                name=uf,
                marker=dict(color=color_map[uf]),
                legendgroup=uf,
                showlegend=True,  # apenas um trace de cada UF aparece na legenda
                visible=(pol == poluentes[0]),  # primeiro poluente aparece inicialmente
                text=y_values,
                textposition="outside"
            )
        )
        #uf_legend_shown[uf] = True
        trace_visibility.append(pol)

# Dropdown menu
buttons = []
for pol in poluentes:
    vis = [p == pol for p in trace_visibility]
    buttons.append(dict(
        label=pol,
        method="update",
        args=[{"visible": vis},
              {"title": f"Monitoramento de {pol}"}]
    ))
anos = np.arange(df_grouped['ANOS_MONITORADOS'].min(),
                 df_grouped['ANOS_MONITORADOS'].max() + 1).tolist()

# Layout
fig.update_layout(
    **layout_style,
    #height=600,
    barmode="stack",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-.2,
        xanchor="left",
        x=0,
        title="UF: "
    ),
    xaxis=dict(
        type='linear',
        title="Ano",
        tickmode='array',
        tickvals=anos,       # força mostrar todos os anos
        ticktext=anos
    ),
    xaxis_range=[df_grouped['ANOS_MONITORADOS'].min() - 0.5, 2024.5],
    yaxis=dict(
        type='linear',
        title="N. pontos de monitoramento",
        tickmode='array',
    ),
    margin=dict(r=120),
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        x=1.15,
        y=1.05,
        showactive=True,
        
    )],
    title=f"Monitoramento de {poluentes[0]}"
)

HTML(fig.to_html(include_plotlyjs="cdn"))
